<a href="https://colab.research.google.com/github/HarshaSatyavardhan/cp/blob/main/GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Simpler normalization for both real and fake images [-1, 1]
tfm = transforms.Compose([
    transforms.Resize(28),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_loader = DataLoader(datasets.MNIST('./data', train=True, download=True, transform=tfm), batch_size=128, shuffle=True)

In [ ]:
class DCGenerator(nn.Module):
    def __init__(self, latent_dim=100):
        super(DCGenerator, self).__init__()
        self.main = nn.Sequential(
            # Input is latent_dim vector
            nn.Linear(latent_dim, 256 * 7 * 7),
            nn.BatchNorm1d(256 * 7 * 7),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Unflatten(1, (256, 7, 7)),

            # Upsample to 14x14
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # Upsample to 28x28
            nn.ConvTranspose2d(128, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)

# Initialize with 100-dim noise
model = DCGenerator(latent_dim=100).to('cuda')
print("Convolutional Generator initialized.")

Convolutional Generator initialized.


In [ ]:
class CustomDiscriminator(nn.Module):
    def __init__(self):
        super(CustomDiscriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.main(x)

model2 = CustomDiscriminator().to('cuda')
print("Custom CNN Discriminator initialized.")

Custom CNN Discriminator initialized.


In [ ]:
from tqdm import tqdm

opt_g = torch.optim.Adam(model.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(model2.parameters(), lr=0.0002, betas=(0.5, 0.999))
criterion = nn.BCELoss()

epochs = 10
for epoch in range(epochs):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    for real_imgs, _ in pbar:
        batch_size = real_imgs.size(0)
        real_imgs = real_imgs.to('cuda')
        real_labels = torch.ones(batch_size, 1).to('cuda')
        fake_labels = torch.zeros(batch_size, 1).to('cuda')

        # --- Train Discriminator ---
        opt_d.zero_grad()
        z = torch.randn(batch_size, 100).to('cuda')
        fake_imgs = model(z)

        loss_d_real = criterion(model2(real_imgs), real_labels)
        loss_d_fake = criterion(model2(fake_imgs.detach()), fake_labels)
        loss_d = (loss_d_real + loss_d_fake) / 2
        loss_d.backward()
        opt_d.step()

        # --- Train Generator ---
        # Freeze discriminator
        for p in model2.parameters(): p.requires_grad = False

        opt_g.zero_grad()
        # We want the discriminator to think these are real
        loss_g = criterion(model2(fake_imgs), real_labels)
        loss_g.backward()
        opt_g.step()

        # Unfreeze discriminator
        for p in model2.parameters(): p.requires_grad = True

        pbar.set_postfix(d_loss=loss_d.item(), g_loss=loss_g.item())

Epoch 9: 100%|██████████| 469/469 [00:25<00:00, 18.42it/s, d_loss=0.0934, g_loss=3.03]


In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    noise = torch.randn(16, 100).to('cuda')
    gen_imgs = model(noise).cpu().squeeze()

plt.figure(figsize=(6, 6))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.imshow(gen_imgs[i], cmap='gray')
    plt.axis('off')
plt.tight_layout()
plt.show()